# Step 1

Run the following 3 "cells" by selecting them and pressing "Shift+Enter".

In [ ]:
# Cell 1

import ipywidgets as widgets
from ipywidgets import IntSlider
from ipywidgets import interact, interactive, fixed, interact_manual, Layout
#import IPython.display as display
#from IPython.display import display
from IPython import display

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
%matplotlib inline
import numpy as np

In [ ]:
# Cell 2

#hbar = 1
#m = 1

def get_all_values(sliders):
    return [slider.value for slider in sliders]

def make_plot(hdisplay,ax,hist,ps,counts):
    for i in range(0,len(counts)):
        hist[i].set_height(counts[i])
    hdisplay.update(fig)

class plane_wave:
    def __init__(self, p, m):
        self.p = p # must be in [10^(-25) kg m/s]
        self.m = m # must be in [MeV]
        self.norm = 1.0/np.sqrt(2*np.pi*6.5821) # in [10^8]
        self.lam = 1.05457/self.p # in [nm]
        self.vel = self.p*4.4937/self.m # in [10^(-6) nm/s)]
        #self.norm = 1.0/np.sqrt(2*np.pi*hbar)
        #self.lam = hbar/self.p
        #self.vel = self.p/(2*self.m)
        
    
    
    def value(self,x,t):
        # t has to be in [10^(-6) s]
        arg = (x-self.vel*t)/self.lam
        return self.norm*np.cos(arg)

In [ ]:
# Cell 3

# Define momentum values in units of [x10^(-25) kg m/s ]
ps = [40+x for x in range(0,10,1)]

# Define sliders
sliders = [0]*len(ps)
for i in range(0,len(ps)):
    sliders[i] = widgets.IntSlider(
        value=5,
        min=0,
        max=10,
        orientation='vertical',
        description='p'+str(i),
        readout=False
    )

# Arrange them side-by-side using HBox (Horizontal Box)
ui_sliders = widgets.HBox(
    sliders,
    layout=Layout(padding="0px 90px")
)

# Initial histogram
fig,ax = plt.subplots(figsize=(10.5, 3))
hdisplay = display.display(fig, display_id=True)
ax.set_ylim(0,10)
ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
ax.yaxis.set_major_locator(ticker.MultipleLocator(1))
ax.set_ylabel(r'$c_i$ (not normalized)')
ax.set_xlabel(r'$p_i$ [$10^{-25}$ kg m/s]')
counts = get_all_values(sliders)
hist = ax.bar(ps,counts,color="blue")
hdisplay.update(fig)

# Display the container
display.display(ui_sliders)

# Define update function for the 3D scatter plot
def update_plot(change):
    make_plot(hdisplay,ax,hist,ps,get_all_values(sliders))

# Attach the update function to the sliders' value change event
for slider in sliders:
    slider.observe(update_plot,'value')

plt.close()

# Step 2

Change the slide bars above that modify the amplitudes of the momentum space wave function $\phi(p)$, which in this case is a finite superposition (sum) of 10 momentum eigenstates. These are basically 10 plane, waves each with a different momentum (that defines its wavelentgh and velocity). You can set some of the coefficients to zero, leave only one plane wave, and make various combinations. Once you have made a choice, then execute the next cell to calculate the properly normalized amplitudes.

In [ ]:
# Normalize coefficients
counts = get_all_values(sliders)
tmp_sum = 0
for i in range(0,len(counts)):
    tmp_sum = tmp_sum + counts[i]**2
coeffs = [ x/np.sqrt(tmp_sum) for x in counts ]

for i in range(0,len(coeffs)):
    print(f"p{i} = {ps[i]}, c{i} = {coeffs[i]:.3f}")
    
# Check normalization
dum = 0
for i in range(0,len(coeffs)):
    dum = dum + coeffs[i]**2
print()
print(f'Check: The sum of the squares of the amplitudes is: {dum:.3f}  (should be 1)')

# Step 3

Set the mass of your particle and plot the corresponding wave function in position representation for $t=0$.

In [ ]:
# Mass in [eV]
m = 0.511


# You don't need to change anything else in this cell below this line

# Create superposition
waves = [ plane_wave(p,m) for p in ps ]

def get_superposition(xs,t):
    ys = np.zeros(N)
    for i in range(0,N):
        for k in range(0,len(waves)):
            ys[i] = ys[i] + coeffs[k]*waves[k].value(xs[i],t)
    return ys

# Plot superposition
xmin = 0 # in [nm]
xmax = 10 # in [nm]
N = 3000 # number of points to use for plotting the wave


from matplotlib import animation
#from IPython.display import HTML

from matplotlib import rc
rc('animation', html='html5') # equivalent to rcParams['animation.html'] = 'html5'


fig_anim,ax = plt.subplots(figsize=(10, 3))
#hdisplay = display.display(fig, display_id=True)
ax.set_xlim(xmin,xmax)
ax.set_ylim(-1,1)
ax.set_ylabel(r'$Re(\psi)$ (not normalized)')
ax.set_xlabel(r'$x$ [$10^{-9}$ m]')
time_step = ax.text(0.5,0.9,r't=0 [$10^{-6}$ s]',transform=ax.transAxes)


xs = np.linspace(xmin,xmax,N)
ys = get_superposition(xs,0)
wave, = ax.plot(xs,ys, 'b', lw=1)

# Step 4

Plot the time evolved state for some time range. Note the 'carrier' and 'envelope' waves. Return to **Step 2**, change the amplitudes of the momentum wave function and repeat steps 3 and 4 to see how the wave patterns change. Try the following experiments:
- Create a single plane wave (fixed momentum, known with 100% probability).
- Create a state that is as well-localized in space as possible.

In the latter case, you may notice that you can get close but never really there. To actually achieve a well-localized wave pattern that "looks" like a particle, you will need a continuous superposition of momenta and not just the 10 values (eigenstates) that we are using here.

In [ ]:
N_frames = 100 # number of frames in the animation

def drawframe(n):
    t = 0.00005*n # in [10^(-6) s]
    ys = get_superposition(xs,t)
    time_step.set_text(rf"t={1000*t:.2f} [$10^{{-9}}$ s]")
    wave.set_data(xs,ys)
    return (wave,wave,time_step)

anim = animation.FuncAnimation(fig_anim, drawframe, frames=N_frames, interval=100, blit=True)

#HTML(anim.to_html5_video())
anim